In [0]:
dbutils.widgets.removeAll() 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog", "aerolineas_dev")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("storageName", "staerolineasdeveus01")

In [0]:

container = dbutils.widgets.get("container")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storageName = dbutils.widgets.get("storageName")

Ingesta dimensional para la tabla bbdd_dim_aerolineas

In [0]:
# definir rutas
path_inicio = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
path_target = f"{catalog}.{schema}."

In [0]:
ruta_csv_dim_aerolinea = f"{path_inicio}dim_aerolineas.csv"
ruta_csv_dim_aeropuertos = f"{path_inicio}dim_aeropuertos.csv"
ruta_csv_dim_aeronaves = f"{path_inicio}dim_aeronaves.csv"
ruta_csv_dim_rutas = f"{path_inicio}dim_rutas.csv"

In [0]:
aerolineas_schema = StructType(fields=[
    StructField("aerolinea_id", StringType(), False),
    StructField("nombre_aerolinea", StringType(), False),
    StructField("pais_sede", StringType(), False),
    StructField("hub_iata", StringType(), False),
    StructField("modelo_negocio", StringType(), False),
    StructField("fecha_inicio_operaciones", DateType(), False),
    StructField("activo", IntegerType(), False)
])

In [0]:
df_dim_aerolineas = spark.read.format("csv") \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(aerolineas_schema) \
    .load(f"{path_inicio}dim_aerolineas.csv")

In [0]:
df_dim_aerolineas_bronze= df_dim_aerolineas.select(
    "aerolinea_id",
    "nombre_aerolinea",
    "pais_sede",
    "hub_iata",
    "modelo_negocio",
    "fecha_inicio_operaciones",
    "activo",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_dim_aerolinea).alias("archivo_origen")
)

In [0]:
df_dim_aerolineas_bronze.write.mode("overwrite").option("overwriteSchema", "true").insertInto(f"{catalog}.{schema}.bbdd_dim_aerolineas")

%md
Ingesta dimensional para la tabla bbdd_dim_aeropuertos

In [0]:
aeropuertos_schema = StructType(fields=[
    StructField("aeropuerto_iata", StringType(), False),
    StructField("nombre_aeropuerto", StringType(), False),
    StructField("ciudad", StringType(), False),
    StructField("pais", StringType(), False),
    StructField("codigo_pais", StringType(), False),
    StructField("zona_horaria", StringType(), False),
    StructField("altitud_m", IntegerType(), False),
    StructField("latitud", DecimalType(10, 6), False),
    StructField("longitud", DecimalType(10, 6), False)
])


df_dim_aeropuertos = (
    spark.read.format("csv")
    .option("header", True)
    .schema(aeropuertos_schema)
    .load(ruta_csv_dim_aeropuertos)
)

df_dim_aeropuertos_bronze = df_dim_aeropuertos.select(
    "aeropuerto_iata",
    "nombre_aeropuerto",
    "ciudad",
    "pais",
    "codigo_pais",
    "zona_horaria",
    "altitud_m",
    "latitud",
    "longitud",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_dim_aeropuertos).alias("archivo_origen")
)

df_dim_aeropuertos_bronze.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.bbdd_dim_aeropuertos")

Ingesta dimensional para la tabla bbdd_dim_aeronaves

In [0]:
aeronaves_schema = StructType(fields=[
    StructField("aeronave_id", StringType(), False),
    StructField("aerolinea_id", StringType(), False),
    StructField("matricula", StringType(), False),
    StructField("modelo", StringType(), False),
    StructField("categoria", StringType(), False),
    StructField("capacidad_total", IntegerType(), False),
    StructField("asientos_business", IntegerType(), False),
    StructField("asientos_economy", IntegerType(), False),
    StructField("tipo_combustible", StringType(), False),
    StructField("anio_fabricacion", IntegerType(), False),
    StructField("estado_aeronave", StringType(), False)
])


df_dim_aeronaves = (
    spark.read.format("csv")
    .option("header", True)
    .schema(aeronaves_schema)
    .load(ruta_csv_dim_aeronaves)
)

df_dim_aeronaves_bronze = df_dim_aeronaves.select(
    "aeronave_id",
    "aerolinea_id",
    "matricula",
    "modelo",
    "categoria",
    "capacidad_total",
    "asientos_business",
    "asientos_economy",
    "tipo_combustible",
    "anio_fabricacion",
    "estado_aeronave",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_dim_aeronaves).alias("archivo_origen")
)

df_dim_aeronaves_bronze.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.bbdd_dim_aeronaves")

Ingesta dimensional para la tabla bbdd_dim_rutas

In [0]:
rutas_schema = StructType(fields=[
    StructField("ruta_id", StringType(), False),
    StructField("origen_iata", StringType(), False),
    StructField("destino_iata", StringType(), False),
    StructField("distancia_km", IntegerType(), False),
    StructField("duracion_programada_min", IntegerType(), False),
    StructField("tipo_ruta", StringType(), False),
    StructField("activa", IntegerType(), False)
])

df_dim_rutas = (
    spark.read.format("csv")
    .option("header", True)
    .schema(rutas_schema)
    .load(ruta_csv_dim_rutas)
)

df_dim_rutas_bronze = df_dim_rutas.select(
    "ruta_id",
    "origen_iata",
    "destino_iata",
    "distancia_km",
    "duracion_programada_min",
    "tipo_ruta",
    "activa",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_dim_rutas).alias("archivo_origen")
)

df_dim_rutas_bronze.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.bbdd_dim_rutas")